# QHRP Hardware Trial

Este notebook no modifica el flujo principal. Sirve para demostrar, de forma acotada, que la parte cuántica del método QHRP puede ejecutarse en backend IBM real y conservar estructura de clustering comparable al simulador ideal.

Qué demuestra para el paper:
- Viabilidad de ejecución fuera del simulador ideal.
- Robustez estructural de la matriz de distancias ordenada.
- Diferencia conceptual entre observables ideales (densidades exactas) y observables reales (distribuciones medidas con shots).

## Requisitos

1. Tener `qiskit-ibm-runtime` instalado en el entorno activo.
2. Haber guardado credenciales de IBM Quantum (token) en tu máquina.
3. Ejecutar con subconjunto pequeño para controlar tiempos de cola/coste.

In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from qiskit_ibm_runtime import QiskitRuntimeService

sys.path.append(os.path.abspath("./src"))
from quantum_hrp import average_density_matrix, compute_distance_matrix, quantum_ordering
from quantum_hrp_hardware import (
    HardwareConfig,
    compute_asset_distributions_hardware,
    compute_distribution_distance_matrix,
    quantum_ordering_from_distance,
)

In [ ]:
# Carga de datos (igual que en el notebook principal)
returns = pd.read_csv("../../data/returns.csv", index_col=0, parse_dates=True)
features = np.load("../../data/features.npy")

# Limpieza mínima para sincronizar activos
orig_cols = returns.columns.tolist()
returns = returns.apply(pd.to_numeric, errors="coerce").dropna(axis=0, how="any").dropna(axis=1, how="all")
std = returns.std(axis=0)
returns = returns.loc[:, std > 1e-8]
surviving_idx = [orig_cols.index(c) for c in returns.columns]
features = features[surviving_idx]

print("returns:", returns.shape)
print("features:", features.shape)

In [ ]:
# Subconjunto acotado para ejecución real
alpha = 2.0
N_hw = min(8, features.shape[0])
T_hw = min(20, features.shape[1])
features_hw = features[:N_hw, :T_hw, :]
P_hw = features_hw.shape[2]

service = QiskitRuntimeService(channel="ibm_quantum")

# SOLO BACKEND REAL (sin fallback).
# Si no hay backend disponible, esta celda debe fallar para dejar claro que no se ejecutó en simulador.
backend = service.least_busy(
    operational=True,
    simulator=False,
    min_num_qubits=P_hw,
)

print(f"Backend real seleccionado: {backend.name}")
print(f"N={N_hw}, T={T_hw}, P={P_hw}")

In [ ]:
# Ejecución hardware: distribuciones medidas -> distancia -> ordenación
cfg = HardwareConfig(shots=512, alpha=alpha, optimization_level=1)

t0 = time.time()
dist_hw_list = compute_asset_distributions_hardware(
    features_hw,
    backend=backend,
    config=cfg,
)
D_quantum_hardware = compute_distribution_distance_matrix(dist_hw_list, metric="hellinger")
order_hw = quantum_ordering_from_distance(D_quantum_hardware)
D_quantum_hardware_ord = D_quantum_hardware[np.ix_(order_hw, order_hw)]
elapsed_hw = time.time() - t0

print(f"Tiempo total hardware: {elapsed_hw:.2f}s | shots={cfg.shots}")

In [ ]:
# Referencia simulador ideal en el mismo subconjunto (comparación estructural)
rho_sim_list = [average_density_matrix(features_hw[i], alpha=alpha) for i in range(N_hw)]
D_quantum_sim = compute_distance_matrix(rho_sim_list)
order_sim = quantum_ordering(D_quantum_sim)
D_quantum_sim_ord = D_quantum_sim[np.ix_(order_sim, order_sim)]

vmax = np.percentile(D_quantum_hardware[np.triu_indices_from(D_quantum_hardware, 1)], 97)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
sns.heatmap(D_quantum_hardware_ord, cmap="coolwarm", vmin=0, vmax=vmax, square=True, cbar=True, ax=axes[0])
axes[0].set_title("IBM QPU ordenado")
axes[0].tick_params(labelsize=7)

sns.heatmap(D_quantum_sim_ord, cmap="coolwarm", vmin=0, vmax=vmax, square=True, cbar=True, ax=axes[1])
axes[1].set_title("Simulador ideal ordenado")
axes[1].tick_params(labelsize=7)

plt.suptitle("Comparación estructural QHRP: hardware real vs simulador ideal")
plt.tight_layout()
plt.show()